# First Part of Experimentation 

For this part we are using text emotion recognition dataset from: https://www.kaggle.com/datasets/bhavikjikadara/emotions-dataset. The dataset provides approximately 400,000 texts scraped from twitter classified into six categories: sadness (0), joy (1), love (2), anger (3), fear (4), and surprise (5).

In [1]:
import pandas as pd
import numpy as np

In [2]:
#loading the datasets
emotion_train = pd.read_csv('data/emotion_train.csv')
emotion_val = pd.read_csv('data/emotion_val.csv')
emotion_test = pd.read_csv('data/emotion_test.csv')

### 1. Testing emotion recognition using a simple BERT model

This part uses the BERT 'base-uncased' model with fully-connected linear classification head to predict labels. By finetuning the BERT model on our dataset we are able to learn accurate representations without the need for very large training datasets / time.

In [3]:
#importing necessary classes from Simple_BERT.py

from Simple_BERT import SimpleBERT, BertFeatureExtractor, MemoryMappedDataset
from torch.utils.data import DataLoader
import torch

/Users/adityakumarpugalia/Desktop/NTU Resources/SC4001-NNDL/Text Emotion Recognition/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/adityakumarpugalia/Desktop/NTU Resources/SC4001-NNDL/Text Emotion Recognition/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


saving bert embeddings of texts

In [4]:
mem = torch.mps.current_allocated_memory()
print(f"Memory allocated: {mem / (1024 ** 2):.2f} MB")

import gc
gc.collect()
torch.mps.empty_cache()
mem = torch.mps.current_allocated_memory()
print(f"Memory allocated: {mem / (1024 ** 2):.2f} MB")

Memory allocated: 0.00 MB
Memory allocated: 0.00 MB


In [5]:
bert_embedder = BertFeatureExtractor()
print("Embedding train data...")
bert_embedder.save_bert_features(emotion_train['text'].tolist(), emotion_train['label'].to_list(), 'data/bert/bert_train_embed.npy', 'data/bert/bert_train_labels.npy')
print("Embedding val data...")
bert_embedder.save_bert_features(emotion_val['text'].to_list(), emotion_val['label'].to_list(), 'data/bert/emotion_val_embed.npy', 'data/bert/emotion_val_labels.npy')
print("Embedding test data...")
bert_embedder.save_bert_features(emotion_test['text'].to_list(), emotion_test['label'].to_list(), 'data/bert/emotion_test_embed.npy', 'data/bert/emotion_test_labels.npy')

Embedding train data...


100%|██████████| 1970/1970 [20:39<00:00,  1.59it/s]


Embedding val data...


100%|██████████| 493/493 [05:07<00:00,  1.61it/s]


Embedding test data...


100%|██████████| 616/616 [06:21<00:00,  1.62it/s]


In [6]:
mem = torch.mps.current_allocated_memory()
print(f"Memory allocated: {mem / (1024 ** 2):.2f} MB")

import gc
gc.collect()
torch.mps.empty_cache()
mem = torch.mps.current_allocated_memory()
print(f"Memory allocated: {mem / (1024 ** 2):.2f} MB")

Memory allocated: 253.16 MB
Memory allocated: 253.16 MB


In [7]:

# load the precomputed features
emotion_bert_train = MemoryMappedDataset('data/bert/bert_train_embed.npy', 'data/bert/bert_train_labels.npy')
emotion_bert_val = MemoryMappedDataset('data/bert/emotion_val_embed.npy', 'data/bert/emotion_val_labels.npy')
emotion_bert_test = MemoryMappedDataset('data/bert/emotion_test_embed.npy', 'data/bert/emotion_test_labels.npy')
# set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)
# create DataLoader for each dataset
emotion_bert_train_data = DataLoader(emotion_bert_train, batch_size=256, shuffle=True)
emotion_bert_val_data = DataLoader(emotion_bert_val, batch_size=256, shuffle=False)
emotion_bert_test_data = DataLoader(emotion_bert_test, batch_size=256, shuffle=False)



In [8]:
mem = torch.mps.current_allocated_memory()
print(f"Memory allocated: {mem / (1024 ** 2):.2f} MB")

Memory allocated: 253.16 MB


In [9]:
print(f"{emotion_bert_train_data.dataset.__getitem__(0)}")  

(tensor([ 8.8646e-02,  5.2148e-04, -4.5784e-03, -1.8514e-01, -1.0020e-01,
        -2.0530e-01,  1.7106e-01,  2.7282e-01, -9.8220e-02, -2.6441e-01,
        -5.7580e-02,  6.8874e-02, -3.4773e-02,  2.8255e-01,  1.4355e-01,
         2.5569e-01, -4.8422e-02,  1.4977e-01,  9.9690e-02, -9.2968e-02,
         8.9565e-02, -3.4172e-01,  1.6324e-01,  2.3598e-01,  3.1136e-02,
        -6.4497e-02,  1.8800e-01,  3.1414e-02,  4.2019e-02, -1.5527e-01,
         1.1599e-01,  8.7760e-02, -2.6765e-01,  1.5867e-01, -1.7926e-02,
         2.8733e-02, -1.9126e-02, -8.4104e-03, -3.1603e-02, -2.7837e-02,
        -1.2480e-01, -1.7884e-02, -6.1897e-02,  3.4635e-02, -1.0085e-02,
        -1.2912e-01, -2.2879e+00, -1.1390e-01, -1.8933e-01, -2.2949e-01,
         3.7605e-01,  1.1770e-01,  3.0440e-01,  2.6368e-01,  2.7337e-01,
         2.2755e-01, -3.0492e-01,  2.7686e-01, -3.7110e-02,  7.6540e-02,
         1.7490e-01, -9.5442e-02, -5.0817e-02, -3.1767e-02, -3.0726e-04,
         4.3787e-02, -9.4356e-03,  6.5456e-02, -1.

/Users/adityakumarpugalia/Desktop/NTU Resources/SC4001-NNDL/Text Emotion Recognition/Simple_BERT.py:43: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:209.)
  emb = torch.from_numpy(self.embeddings[idx]).float()


In [10]:
#train model
torch.mps.empty_cache()
emotion_model = SimpleBERT(num_labels=6)
train_accuracies, train_losses, val_accuracies, val_losses = emotion_model.train_data(train_loader= emotion_bert_train_data, val_loader= emotion_bert_val_data, epochs= 100, patience= 3)

Epoch 1/100
Epoch 1/100 - Training Loss: 1.1877333189490513 - Training Accuracy: 0.5606578190402507
Epoch 1/100 - Validation Loss: 1.0731046577638985 - Validation Accuracy: 0.6048054338856091
Saved best model at epoch 1
Epoch 2/100
Epoch 2/100 - Training Loss: 1.0714311414494806 - Training Accuracy: 0.6050492450572436
Epoch 2/100 - Validation Loss: 1.0260249737110743 - Validation Accuracy: 0.6201834571192789
Saved best model at epoch 2
Epoch 3/100
Epoch 3/100 - Training Loss: 1.0400822867178963 - Training Accuracy: 0.6161361277483122
Epoch 3/100 - Validation Loss: 1.0046685019205663 - Validation Accuracy: 0.6344823208277789
Saved best model at epoch 3
Epoch 4/100
Epoch 4/100 - Training Loss: 1.0248298176475568 - Training Accuracy: 0.6236109656347899
Epoch 4/100 - Validation Loss: 0.9890751081868171 - Validation Accuracy: 0.6342760109185552
Saved best model at epoch 4
Epoch 5/100
Epoch 5/100 - Training Loss: 1.0167355852626347 - Training Accuracy: 0.624521911908004
Epoch 5/100 - Validat

In [11]:
#test model
torch.mps.empty_cache()
test_accuracy, test_loss = emotion_model.evaluate(test_loader= emotion_bert_test_data)
print(f"Test Accuracy: {test_accuracy}")
print(f"Test Loss: {test_loss}")


Test Accuracy: 0.6504157938170507
Test Loss: 0.9523399358672626


In [12]:
# save results as python objects
import pickle
with open('results/emotion_bert_results.pkl', 'wb') as f:
    pickle.dump((train_accuracies, train_losses, val_accuracies, val_losses, test_accuracy, test_loss), f)

### Text Emotion Recognition using CNN

In [6]:
from Simple_BERT import DistilBERTDataset

emotion_CNN_train = DistilBERTDataset(emotion_train['text'].tolist(), emotion_train['label'].to_list())
emotion_CNN_val = DistilBERTDataset(emotion_val['text'].tolist(), emotion_val['label'].to_list())
emotion_CNN_test = DistilBERTDataset(emotion_test['text'].tolist(), emotion_test['label'].to_list())

# set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# create DataLoader for each dataset
emotion_CNN_train_data = DataLoader(emotion_CNN_train, batch_size=128, shuffle=True)
emotion_CNN_val_data = DataLoader(emotion_CNN_val, batch_size=128, shuffle=False)
emotion_CNN_test_data = DataLoader(emotion_CNN_test, batch_size=128, shuffle=False)

In [5]:
#train model
from BERT_CNN import BertCNN
torch.mps.empty_cache()
emotion_CNN_model = BertCNN(num_labels = 6)
train_accuracies, train_losses, val_accuracies, val_losses = emotion_CNN_model.train_CNN(train_dataloader= emotion_CNN_train_data, val_dataloader= emotion_CNN_val_data, num_epochs= 100, patience= 1)


NameError: name 'emotion_CNN_train_data' is not defined

In [9]:
emotion_CNN_model = BertCNN(6)
emotion_CNN_model.load_state_dict(torch.load('models/best_CNN_model.pt'))

<All keys matched successfully>

In [ ]:
test_predictions, test_logits,test_loss, test_accuracy  = emotion_CNN_model.evaluate(emotion_CNN_test_data)
print(test_logits[:10], test_predictions[:10], test_accuracy, test_loss)

In [ ]:
# save results as python objects
import pickle
with open('results/emotion_CNN_results.pkl', 'wb') as f:
    pickle.dump((train_accuracies, train_losses, val_accuracies, val_losses, test_predictions, test_logits, test_accuracy, test_loss), f)